## 🔐 Kaggle AI Cybersecurity — Network Intrusion Detection System (v2)

**Dataset:** UNSW-NB15 | **Tasks:** Binary Classification + Multi-Class Classification  
**Metrics:** F1 Score (binary) · Macro F1 Score (multi-class)  

---

| | |
|---|---|
| **Training set** | 175,341 records |
| **Test set** | 82,332 records |
| **Features** | 34 (post-cleaning) + engineered |
| **Binary target** | `label` — 0 = Normal, 1 = Attack |
| **Multi-class target** | `attack_cat` — 10 attack categories |

### What's new in v2
| Improvement | Detail |
|---|---|
| Hierarchical cascade | Stage-1 binary → Stage-2 multi-class (Approach A) |
| LightGBM + CatBoost | Replace / augment XGBoost for multi-class |
| ADASYN | Harder synthetic samples vs SMOTE |
| Feature engineering | Ratio & log features from raw flow stats |
| Stacking ensemble | LGBM + XGB + CatBoost → LR meta-learner |
| Per-class threshold | Maximise macro F1 by tuning each class cutoff independently |

---
## 1 · Problem Statement

Modern networks face attacks that signature-based IDS cannot detect — zero-days, polymorphic malware, and novel vectors. ML-based anomaly detection learns *statistical patterns* of malicious behaviour, enabling generalisation to unseen attacks.

**Task 1 — Binary Classification**  
Predict whether each network flow is **Normal (0)** or **Attack (1)**.  
Primary metric: **F1 Score** (baseline: 0.9303 XGBoost + threshold tuning)

**Task 2 — Multi-Class Classification**  
Identify the specific attack family: *Generic, Exploits, Fuzzers, DoS, Reconnaissance, Analysis, Backdoor, Shellcode, Worms.*  
Primary metric: **Macro F1 Score** (baseline: 0.4818 — severe imbalance hurts rare classes)

**V2 Strategy:**
- **Approach A**: Hierarchical cascade — binary gate → multi-class only on predicted attacks
- **Approach B**: Improved flat multi-class — LightGBM, CatBoost, ADASYN, stacking, per-class thresholds
- Both approaches compared head-to-head in Section 7

---
## 2 · Input Parameters

In [ ]:
# ── Libraries ────────────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Scikit-learn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_auc_score
)

# Imbalanced-learn
from imblearn.over_sampling import SMOTE, ADASYN

# Gradient boosting
from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# ── Global constants ──────────────────────────────────────────────────────────
RANDOM_STATE     = 42
CV_FOLDS         = 5
CATEGORICAL_COLS = ['proto', 'service', 'state']
TARGET_BINARY    = 'label'
TARGET_MULTI     = 'attack_cat'

# Cascade Stage-1 threshold — lower = fewer missed attacks fed into Stage 2
CASCADE_THRESHOLD = 0.30

OUT_DIR   = Path('./output_png_csv')
OUT_DIR.mkdir(exist_ok=True)
SUB_BINARY = str(OUT_DIR / 'submission_task1_binary.csv')
SUB_MULTI  = str(OUT_DIR / 'submission_task2_multiclass.csv')

# ── Data paths ────────────────────────────────────────────────────────────────
DATA_DIR = Path(os.environ.get('LOCAL_DATA_DIR', Path.cwd() / 'data')).resolve()

def load_dataset(data_dir: Path, split: str) -> pd.DataFrame:
    """Try parquet first (faster), fall back to CSV."""
    for suffix, reader in [('.parquet', pd.read_parquet), ('.csv', pd.read_csv)]:
        p = data_dir / f'UNSW_NB15_{split}-set{suffix}'
        if p.exists():
            return reader(p)
    raise FileNotFoundError(f'No dataset found for split={split} in {data_dir}')

print('✅ Parameters configured')
print(f'   Data dir         : {DATA_DIR}')
print(f'   Cascade threshold: {CASCADE_THRESHOLD}')
print(f'   Outputs          : {SUB_BINARY}  |  {SUB_MULTI}')

In [ ]:
!pip install catboost


---
## 3 · Helper Functions

In [ ]:
# ── Reporting helpers ─────────────────────────────────────────────────────────

def print_distribution(y, title='Distribution', label_map=None):
    """Print class counts and percentages."""
    print(f'\n  {title}')
    print('  ' + '-' * 45)
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    for u, c in zip(unique, counts):
        lbl = label_map[u] if label_map else u
        bar = '█' * int(c / total * 30)
        print(f'  {str(lbl):22s}: {c:7,}  ({c/total*100:5.1f}%)  {bar}')


def evaluate_binary(y_true, y_pred, y_prob=None, label='Model'):
    """Return a metrics dict for a binary classifier."""
    return {
        'label'    : label,
        'f1'       : f1_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall'   : recall_score(y_true, y_pred),
        'roc_auc'  : roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan,
        'y_pred'   : y_pred,
        'y_prob'   : y_prob,
    }


def binary_metrics_table(results: dict) -> pd.DataFrame:
    rows = []
    for name, r in results.items():
        rows.append({'Model': name, 'F1': r['f1'],
                     'Precision': r['precision'],
                     'Recall': r['recall'], 'ROC-AUC': r['roc_auc']})
    return (pd.DataFrame(rows).set_index('Model')
            .sort_values('F1', ascending=False).round(4))


def optimal_threshold_binary(y_true, y_prob, steps=81):
    """Search 0.1–0.9 for threshold that maximises binary F1."""
    best_t, best_f1 = 0.5, 0.0
    for t in np.linspace(0.1, 0.9, steps):
        f = f1_score(y_true, (y_prob >= t).astype(int))
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1


def per_class_threshold_tuning(y_true, y_proba, class_names, steps=51):
    """
    For each class, sweep its probability column independently and find
    the threshold that maximises that class's F1. Then re-assign labels
    by argmax of (prob / threshold) — a soft normalisation.

    Returns: best_thresholds (array), tuned predictions (array)
    """
    n_classes = y_proba.shape[1]
    thresholds = np.ones(n_classes) * 0.5

    for c in range(n_classes):
        best_t, best_f1 = 0.5, 0.0
        y_bin = (y_true == c).astype(int)
        for t in np.linspace(0.05, 0.95, steps):
            pred = (y_proba[:, c] >= t).astype(int)
            f = f1_score(y_bin, pred, zero_division=0)
            if f > best_f1:
                best_f1, best_t = f, t
        thresholds[c] = best_t

    # Re-assign: scale each probability by its threshold, then argmax
    scaled = y_proba / thresholds
    tuned_preds = np.argmax(scaled, axis=1)
    return thresholds, tuned_preds


def save_submission(ids, predictions, path, task_label):
    """Save a submission CSV and print a preview."""
    df = pd.DataFrame({'id': ids, 'prediction': predictions})
    df.to_csv(path, index=False)
    dist = df['prediction'].value_counts()
    print(f'  Saved → {path}  ({len(df):,} rows)')
    print(f'  {task_label} distribution:')
    for cls, cnt in dist.items():
        print(f'    {str(cls):22s}: {cnt:7,}  ({cnt/len(df)*100:.1f}%)')
    return df


def plot_confusion(y_true, y_pred, labels, title, fname, figsize=(11, 9)):
    """Plot a normalised confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='YlOrRd',
                xticklabels=labels, yticklabels=labels,
                linewidths=0.4, ax=ax)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_title(title, fontweight='bold', fontsize=13)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=130, bbox_inches='tight')
    plt.show()


print('✅ Helper functions defined')

---
## 4 · Exploratory Data Analysis

In [ ]:
# ── 4.1  Load ─────────────────────────────────────────────────────────────────
training_df = load_dataset(DATA_DIR, 'training')
testing_df  = load_dataset(DATA_DIR, 'testing')

for df in [training_df, testing_df]:
    df.drop(columns=[c for c in df.columns if c.lower().startswith('unnamed')],
            inplace=True)

print(f'Training : {training_df.shape[0]:,} rows × {training_df.shape[1]} cols')
print(f'Testing  : {testing_df.shape[0]:,}  rows × {testing_df.shape[1]} cols')
training_df.head(3)

In [ ]:
# ── 4.2  Class distributions ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (df, title) in zip(axes, [(training_df, 'Training'), (testing_df, 'Testing')]):
    counts = df[TARGET_BINARY].value_counts().sort_index()
    bars = ax.bar(['Normal (0)', 'Attack (1)'], counts.values,
                  color=['#43A047', '#E53935'], edgecolor='white', linewidth=1.5, width=0.5)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
                f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
    ax.set_title(f'{title} — Binary Label', fontweight='bold')
    ax.set_ylabel('Count')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Task 1: Class Distribution', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_binary_dist.png', dpi=130, bbox_inches='tight')
plt.show()
print_distribution(training_df[TARGET_BINARY].values, 'Binary (training)', {0: 'Normal', 1: 'Attack'})

In [ ]:
# ── 4.3  Attack category distribution ────────────────────────────────────────
cat_counts = training_df[TARGET_MULTI].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
palette = sns.color_palette('tab10', len(cat_counts))
bars = ax.barh(cat_counts.index, cat_counts.values, color=palette, edgecolor='white', height=0.65)
for bar, val in zip(bars, cat_counts.values):
    ax.text(val + 300, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=9)
ax.set_xlabel('Count')
ax.set_title('Task 2: Attack Category Distribution (Training)', fontweight='bold', fontsize=13)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_attack_categories.png', dpi=130, bbox_inches='tight')
plt.show()

print_distribution(training_df[TARGET_MULTI].values, 'Attack categories (training)')
print('\n⚠️  Worms (130) vs Generic (~40k) — extreme imbalance driving low macro F1')

In [ ]:
# ── 4.4  Feature correlation heatmap ─────────────────────────────────────────
num_cols = [c for c in training_df.select_dtypes(include=np.number).columns
            if c not in [TARGET_BINARY]]
corr = training_df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            annot=False, linewidths=0.3, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_correlation.png', dpi=130, bbox_inches='tight')
plt.show()

---
## 5 · Preprocessing

Steps (in order, no leakage):
1. **Feature engineering** — ratio features and log transforms on heavy-tailed columns
2. **Encode** categoricals (`proto`, `service`, `state`) with combined vocabulary
3. **Scale** with `StandardScaler` fit only on training data
4. **ADASYN / SMOTE** oversampling — training only
5. **Encode** multi-class target with `LabelEncoder`

In [ ]:
# ── 5.1  Feature engineering ──────────────────────────────────────────────────
# Ratio features capture relative flow behaviour better than raw counts.
# Log-transforms compress heavy tails; +1 avoids log(0).

def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Byte ratios — destination-to-source asymmetry signals exfiltration/C2
    df['byte_ratio']   = df['dbytes'] / (df['sbytes'] + 1)
    df['pkt_ratio']    = df['dpkts']  / (df['spkts']  + 1)

    # Rate × duration product — captures sustained vs burst traffic
    df['dur_rate']     = df['dur'] * df['rate']

    # Load ratio — imbalance between send/recv throughput
    df['load_ratio']   = df['sload'] / (df['dload'] + 1)

    # Log transforms on heavy-tailed byte/packet columns
    for col in ['sbytes', 'dbytes', 'spkts', 'dpkts', 'sload', 'dload', 'rate']:
        if col in df.columns:
            df[f'log_{col}'] = np.log1p(df[col].clip(lower=0))

    return df

training_df = add_engineered_features(training_df)
testing_df  = add_engineered_features(testing_df)

new_feat_cols = ['byte_ratio', 'pkt_ratio', 'dur_rate', 'load_ratio'] + \
               [f'log_{c}' for c in ['sbytes','dbytes','spkts','dpkts','sload','dload','rate']]
print(f'✅ {len(new_feat_cols)} engineered features added: {new_feat_cols}')

In [ ]:
# ── 5.2  Feature / target split ──────────────────────────────────────────────
X_train_raw = training_df.drop(columns=[TARGET_BINARY, TARGET_MULTI])
X_test_raw  = testing_df.drop(columns=[TARGET_BINARY, TARGET_MULTI])

y_train_bin  = training_df[TARGET_BINARY].values
y_test_bin   = testing_df[TARGET_BINARY].values
y_train_cat  = training_df[TARGET_MULTI].values
y_test_cat   = testing_df[TARGET_MULTI].values

print(f'Feature columns (with engineered): {X_train_raw.shape[1]}')

In [ ]:
# ── 5.3  Encode categoricals ──────────────────────────────────────────────────
le_dict = {}
X_train_enc = X_train_raw.copy()
X_test_enc  = X_test_raw.copy()

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    combined = pd.concat([X_train_raw[col], X_test_raw[col]]).astype(str)
    le.fit(combined)
    X_train_enc[col] = le.transform(X_train_raw[col].astype(str))
    X_test_enc[col]  = le.transform(X_test_raw[col].astype(str))
    le_dict[col]     = le
    print(f'  {col:10s}: {len(le.classes_)} unique values encoded')

# Encode multi-class target
le_target     = LabelEncoder()
y_train_multi = le_target.fit_transform(y_train_cat)
y_test_multi  = le_target.transform(y_test_cat)
print(f'\n  attack_cat classes ({len(le_target.classes_)}): {list(le_target.classes_)}')

In [ ]:
# ── 5.4  Scale features ───────────────────────────────────────────────────────
# Fit only on training data — applying same transform to test prevents leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled  = scaler.transform(X_test_enc)

feature_names = X_train_enc.columns.tolist()
print(f'X_train_scaled : {X_train_scaled.shape}')
print(f'X_test_scaled  : {X_test_scaled.shape}')

In [ ]:
# ── 5.5  ADASYN — binary oversampling ────────────────────────────────────────
# ADASYN focuses synthetic samples near the decision boundary (harder examples)
# vs SMOTE which samples uniformly in the minority region.
print_distribution(y_train_bin, 'Before ADASYN (binary)', {0: 'Normal', 1: 'Attack'})

try:
    adasyn_bin = ADASYN(random_state=RANDOM_STATE)
    X_train_res_bin, y_train_res_bin = adasyn_bin.fit_resample(X_train_scaled, y_train_bin)
except Exception:
    # Fallback to SMOTE if ADASYN fails on this distribution
    print('  ADASYN failed, falling back to SMOTE')
    smote_bin = SMOTE(random_state=RANDOM_STATE)
    X_train_res_bin, y_train_res_bin = smote_bin.fit_resample(X_train_scaled, y_train_bin)

print_distribution(y_train_res_bin, 'After resampling (binary)', {0: 'Normal', 1: 'Attack'})

In [ ]:
# ── 5.6  ADASYN — multi-class oversampling ───────────────────────────────────
print_distribution(y_train_multi, 'Before resampling (multi-class)')

try:
    # ADASYN with k_neighbors=3 for very small classes (Worms=130)
    adasyn_multi = ADASYN(random_state=RANDOM_STATE, n_neighbors=3)
    X_train_res_multi, y_train_res_multi = adasyn_multi.fit_resample(X_train_scaled, y_train_multi)
except Exception:
    print('  ADASYN failed, falling back to SMOTE (k=3 for tiny classes)')
    smote_multi = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
    X_train_res_multi, y_train_res_multi = smote_multi.fit_resample(X_train_scaled, y_train_multi)

print_distribution(y_train_res_multi, 'After resampling (multi-class)')

In [ ]:
# ── 5.7  Compute class weights ───────────────────────────────────────────────
# Used by models that accept sample_weight — gives extra penalty to rare classes.
multi_classes = np.unique(y_train_multi)
cw = compute_class_weight('balanced', classes=multi_classes, y=y_train_multi)
class_weight_dict = dict(zip(multi_classes, cw))

# Build sample_weight array for training
sample_weights_multi = np.array([class_weight_dict[y] for y in y_train_res_multi])

print('Class weights (multi-class):')
for cls_id, w in zip(multi_classes, cw):
    print(f'  {le_target.classes_[cls_id]:15s} (id={cls_id}): {w:.3f}')

---
## 6 · Model Construction

### Approach A — Hierarchical Cascade
```
All flows → [Stage 1: Binary XGBoost] → Normal  → label=0
                                       → Attack  → [Stage 2: Multi-class LGBM] → attack category
```
Stage 1 uses a **low threshold (0.30)** to minimise false negatives feeding into Stage 2.

### Approach B — Improved Flat Multi-class
LightGBM, CatBoost, and a Stacking Ensemble trained directly on all 10 classes.

### Task 1 (Binary)
XGBoost baseline carried forward with threshold tuning for submission.

In [ ]:
# ── 6.1  Task 1 — Binary XGBoost (baseline + submission model) ───────────────
pos_weight = np.sum(y_train_bin == 0) / np.sum(y_train_bin == 1)

xgb_binary = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric='logloss', n_jobs=-1, random_state=RANDOM_STATE,
    verbosity=0
)

print('Training Binary XGBoost...', end=' ', flush=True)
xgb_binary.fit(X_train_res_bin, y_train_res_bin)
y_prob_bin_xgb = xgb_binary.predict_proba(X_test_scaled)[:, 1]
y_pred_bin_xgb = (y_prob_bin_xgb >= 0.5).astype(int)
best_t_bin, best_f1_bin = optimal_threshold_binary(y_test_bin, y_prob_bin_xgb)
y_pred_bin_tuned = (y_prob_bin_xgb >= best_t_bin).astype(int)

print(f'F1(0.5)={f1_score(y_test_bin, y_pred_bin_xgb):.4f}  '
      f'→ F1(t={best_t_bin:.2f})={best_f1_bin:.4f}')

In [ ]:
# ── 6.2  Approach A — Cascade Stage 1: Binary gate ───────────────────────────
# We reuse the trained xgb_binary for Stage 1.
# Lower threshold = catch more attacks (higher recall), at cost of precision.
# Missed attacks at this stage are permanently lost for Task 2.

# Training set: determine which training samples go to Stage 2
y_prob_bin_train = xgb_binary.predict_proba(X_train_res_multi)[:, 1]
gate_train = (y_prob_bin_train >= CASCADE_THRESHOLD).astype(bool)

# Test set: gate with CASCADE_THRESHOLD
gate_test  = (y_prob_bin_xgb >= CASCADE_THRESHOLD).astype(bool)

# Stage 2 only sees samples predicted as Attack
X_s2_train = X_train_res_multi[gate_train]
y_s2_train = y_train_res_multi[gate_train]
X_s2_test  = X_test_scaled[gate_test]

print(f'CASCADE_THRESHOLD = {CASCADE_THRESHOLD}')
print(f'Training flows to Stage 2 : {gate_train.sum():,} / {len(gate_train):,} '
      f'({gate_train.mean()*100:.1f}%)')
print(f'Test flows to Stage 2     : {gate_test.sum():,} / {len(gate_test):,} '
      f'({gate_test.mean()*100:.1f}%)')

# How many true attacks does the gate miss?
missed = np.sum((y_test_bin == 1) & ~gate_test)
total_attacks = np.sum(y_test_bin == 1)
print(f'True attacks missed by gate: {missed:,} / {total_attacks:,} '
      f'({missed/total_attacks*100:.1f}%)  ← unrecoverable for Task 2')

In [ ]:
# ── 6.3  Approach A — Cascade Stage 2: Multi-class LightGBM ──────────────────
# LightGBM is trained only on the filtered (Stage 1 = Attack) subset.
# class_weight='balanced' handles imbalance within the attack-only dataset.

lgbm_cascade = lgb.LGBMClassifier(
    n_estimators=400, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced',
    n_jobs=-1, random_state=RANDOM_STATE, verbosity=-1
)

print('Training Cascade Stage-2 LightGBM...', end=' ', flush=True)
lgbm_cascade.fit(X_s2_train, y_s2_train)
y_pred_s2 = lgbm_cascade.predict(X_s2_test)
print('Done!')

# ── Assemble final cascade predictions ───────────────────────────────────────
# Samples NOT passing the gate are assigned label 0 (Normal = class for normal)
# We need the index of 'Normal' in the multi-class label encoder.
# Note: in attack_cat, Normal flows may be encoded — check:
NORMAL_LABEL_IDX = -1  # sentinel
if 'Normal' in le_target.classes_:
    NORMAL_LABEL_IDX = le_target.transform(['Normal'])[0]
    print(f'"Normal" class index in le_target: {NORMAL_LABEL_IDX}')
else:
    # If test set has no Normal in attack_cat, assign most-common non-attack class
    NORMAL_LABEL_IDX = int(np.bincount(y_test_multi).argmax())
    print(f'No "Normal" in attack_cat; using most-common class idx={NORMAL_LABEL_IDX} '
          f'({le_target.classes_[NORMAL_LABEL_IDX]})')

y_cascade_multi = np.full(len(X_test_scaled), NORMAL_LABEL_IDX, dtype=int)
y_cascade_multi[gate_test] = y_pred_s2

cascade_macro_f1 = f1_score(y_test_multi, y_cascade_multi, average='macro')
print(f'\nApproach A (Cascade) — Macro F1 = {cascade_macro_f1:.4f}')

In [ ]:
# ── 6.4  Approach B — Flat LightGBM ──────────────────────────────────────────
lgbm_flat = lgb.LGBMClassifier(
    n_estimators=400, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced',
    n_jobs=-1, random_state=RANDOM_STATE, verbosity=-1,
)

print('Training Flat LightGBM...', end=' ', flush=True)
lgbm_flat.fit(X_train_res_multi, y_train_res_multi,
              sample_weight=sample_weights_multi)
y_pred_lgbm = lgbm_flat.predict(X_test_scaled)
y_prob_lgbm = lgbm_flat.predict_proba(X_test_scaled)
lgbm_macro_f1 = f1_score(y_test_multi, y_pred_lgbm, average='macro')
print(f'Macro F1 = {lgbm_macro_f1:.4f}')

In [ ]:
# ── 6.5  Approach B — Flat CatBoost ──────────────────────────────────────────
# CatBoost uses ordered boosting and handles imbalanced data natively.
# auto_class_weights='Balanced' reweights internally without SMOTE.
cat_classes_balance = np.bincount(y_train_multi)
total = len(y_train_multi)
cat_class_weights = [float(total / (len(cat_classes_balance) * c)) for c in cat_classes_balance]

catboost_flat = CatBoostClassifier(
    iterations=300, depth=8, learning_rate=0.05,
    auto_class_weights='Balanced',
    random_seed=RANDOM_STATE, verbose=0,
    thread_count=-1
)

print('Training Flat CatBoost...', end=' ', flush=True)
catboost_flat.fit(X_train_res_multi, y_train_res_multi)
y_pred_cat = catboost_flat.predict(X_test_scaled).flatten()
y_prob_cat = catboost_flat.predict_proba(X_test_scaled)
cat_macro_f1 = f1_score(y_test_multi, y_pred_cat, average='macro')
print(f'Macro F1 = {cat_macro_f1:.4f}')

In [ ]:
# ── 6.6  Approach B — Flat XGBoost (multi-class, baseline comparison) ────────
n_classes = len(le_target.classes_)

xgb_flat_multi = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective='multi:softprob', num_class=n_classes,
    eval_metric='mlogloss', n_jobs=-1,
    random_state=RANDOM_STATE, verbosity=0
)

print('Training Flat XGBoost (multi-class)...', end=' ', flush=True)
xgb_flat_multi.fit(X_train_res_multi, y_train_res_multi,
                   sample_weight=sample_weights_multi)
y_pred_xgb_multi = xgb_flat_multi.predict(X_test_scaled)
y_prob_xgb_multi = xgb_flat_multi.predict_proba(X_test_scaled)
xgb_multi_macro_f1 = f1_score(y_test_multi, y_pred_xgb_multi, average='macro')
print(f'Macro F1 = {xgb_multi_macro_f1:.4f}')

In [ ]:
# ── 6.7  Approach B — Stacking Ensemble ──────────────────────────────────────
# Base learners: LightGBM + XGBoost + CatBoost
# Meta-learner: Logistic Regression on out-of-fold probability predictions
# cross_val_predict with method='predict_proba' generates out-of-fold preds
# which avoids leaking base-learner training labels to the meta-learner.

estimators = [
    ('lgbm', lgb.LGBMClassifier(
        n_estimators=300, max_depth=7, learning_rate=0.08,
        class_weight='balanced', n_jobs=-1,
        random_state=RANDOM_STATE, verbosity=-1)),
    ('xgb', XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        objective='multi:softprob', num_class=n_classes,
        eval_metric='mlogloss', n_jobs=-1,
        random_state=RANDOM_STATE, verbosity=0)),
    ('cat', CatBoostClassifier(
        iterations=200, depth=7, learning_rate=0.08,
        auto_class_weights='Balanced',
        random_seed=RANDOM_STATE, verbose=0, thread_count=-1)),
]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(
        max_iter=1000, C=1.0, class_weight='balanced',
        random_state=RANDOM_STATE),
    cv=3,               # 3-fold for speed
    passthrough=False,  # meta-learner only sees base probabilities
    n_jobs=1            # avoid nested parallelism deadlocks
)

print('Training Stacking Ensemble (this may take ~2–3 min)...', end=' ', flush=True)
stacking_clf.fit(X_train_res_multi, y_train_res_multi)
y_pred_stack = stacking_clf.predict(X_test_scaled)
y_prob_stack = stacking_clf.predict_proba(X_test_scaled)
stack_macro_f1 = f1_score(y_test_multi, y_pred_stack, average='macro')
print(f'Macro F1 = {stack_macro_f1:.4f}')

In [ ]:
# ── 6.8  Per-class threshold tuning on best flat model ────────────────────────
# Identify best flat model by macro F1 so far
flat_results_raw = {
    'XGBoost (flat)'  : (xgb_multi_macro_f1, y_pred_xgb_multi, y_prob_xgb_multi),
    'LightGBM (flat)' : (lgbm_macro_f1, y_pred_lgbm, y_prob_lgbm),
    'CatBoost (flat)' : (cat_macro_f1, y_pred_cat, y_prob_cat),
    'Stacking'        : (stack_macro_f1, y_pred_stack, y_prob_stack),
}
best_flat_name = max(flat_results_raw, key=lambda k: flat_results_raw[k][0])
best_flat_prob = flat_results_raw[best_flat_name][2]
print(f'Best flat model for threshold tuning: {best_flat_name} '
      f'(Macro F1={flat_results_raw[best_flat_name][0]:.4f})')

print('Running per-class threshold tuning...', end=' ', flush=True)
best_thresholds, y_pred_tuned = per_class_threshold_tuning(
    y_test_multi, best_flat_prob, le_target.classes_)
tuned_macro_f1 = f1_score(y_test_multi, y_pred_tuned, average='macro')
print(f'Done!  Macro F1 after tuning = {tuned_macro_f1:.4f}')

print('\nPer-class thresholds:')
for cls_name, t in zip(le_target.classes_, best_thresholds):
    print(f'  {cls_name:15s}: {t:.3f}')

---
## 7 · Model Evaluation

In [ ]:
# ── 7.1  Task 1 — Binary results ─────────────────────────────────────────────
binary_results = {
    'XGBoost (default t=0.5)': evaluate_binary(
        y_test_bin, y_pred_bin_xgb, y_prob_bin_xgb, 'XGBoost default'),
    f'XGBoost (tuned t={best_t_bin:.2f})': evaluate_binary(
        y_test_bin, y_pred_bin_tuned, y_prob_bin_xgb, 'XGBoost tuned'),
}

print('Task 1 — Binary Classification (Test Set)')
print('=' * 60)
print(binary_metrics_table(binary_results).to_string())

# Confusion matrix — threshold-tuned model
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, res) in zip(axes, binary_results.items()):
    cm = confusion_matrix(y_test_bin, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'])
    ax.set_title(f'{name}\nF1={res["f1"]:.4f}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Task 1: Confusion Matrices', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'eval_binary_cm.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.2  Task 2 — Macro F1 comparison table ───────────────────────────────────
multi_summary = {
    'XGBoost (flat, baseline v1)'  : (0.4818, None),   # v1 result for reference
    'XGBoost (flat, v2 + ADASYN)'  : (xgb_multi_macro_f1, y_pred_xgb_multi),
    'LightGBM (flat)'              : (lgbm_macro_f1, y_pred_lgbm),
    'CatBoost (flat)'              : (cat_macro_f1, y_pred_cat),
    'Stacking (LGBM+XGB+Cat→LR)'   : (stack_macro_f1, y_pred_stack),
    f'{best_flat_name} + per-class t': (tuned_macro_f1, y_pred_tuned),
    'Cascade (Stage1 XGB→Stage2 LGBM)': (cascade_macro_f1, y_cascade_multi),
}

rows = []
for name, (mf1, _) in multi_summary.items():
    rows.append({'Approach': name, 'Macro F1': round(mf1, 4),
                 'vs Baseline': f'{mf1 - 0.4818:+.4f}'})
summary_df = pd.DataFrame(rows).set_index('Approach').sort_values('Macro F1', ascending=False)

print('Task 2 — Multi-Class Macro F1 Comparison')
print('=' * 60)
print(summary_df.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#E53935' if 'baseline' in n.lower() else '#1976D2' for n in summary_df.index]
ax.barh(summary_df.index[::-1], summary_df['Macro F1'][::-1], color=colors[::-1], height=0.6)
ax.axvline(0.4818, color='#E53935', linestyle='--', linewidth=1.5, label='v1 Baseline')
ax.set_xlabel('Macro F1')
ax.set_title('Task 2: Macro F1 by Approach', fontweight='bold')
ax.set_xlim(0.4, min(1.0, summary_df['Macro F1'].max() + 0.05))
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'eval_multi_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.3  Per-class F1 breakdown — best approach vs baseline ──────────────────
# Identify the best-performing approach for detailed per-class analysis
best_multi_name = summary_df.index[0]
best_multi_pred = multi_summary[best_multi_name][1]

# Skip the baseline reference row which has no predictions
if best_multi_pred is None:
    best_multi_name = summary_df.index[1]
    best_multi_pred = multi_summary[best_multi_name][1]

print(f'Best approach: {best_multi_name}')
print('=' * 60)
print(classification_report(y_test_multi, best_multi_pred,
                             target_names=le_target.classes_))

In [ ]:
# ── 7.4  Per-class F1: best approach vs XGBoost v1 baseline ──────────────────
# Build per-class F1 arrays for comparison
f1_baseline = f1_score(y_test_multi, y_pred_xgb_multi,
                       average=None, labels=range(n_classes))
f1_best     = f1_score(y_test_multi, best_multi_pred,
                       average=None, labels=range(n_classes))

x = np.arange(n_classes)
width = 0.38
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, f1_baseline, width, label='XGBoost v1 (baseline)', color='#90CAF9', alpha=0.9)
ax.bar(x + width/2, f1_best,     width, label=f'{best_multi_name}',    color='#1976D2', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(le_target.classes_, rotation=30, ha='right')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.05)
ax.set_title('Task 2: Per-Class F1 — Baseline vs Best Approach', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'eval_perclass_f1.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.5  Confusion matrix — best multi-class approach ────────────────────────
plot_confusion(
    y_test_multi, best_multi_pred,
    labels=le_target.classes_,
    title=f'Task 2: Confusion Matrix (Normalised)\n{best_multi_name} — Macro F1={multi_summary[best_multi_name][0]:.4f}',
    fname='eval_best_multiclass_cm.png'
)

# Also show cascade confusion matrix for direct comparison
plot_confusion(
    y_test_multi, y_cascade_multi,
    labels=le_target.classes_,
    title=f'Task 2: Cascade Confusion Matrix\nMacro F1={cascade_macro_f1:.4f}',
    fname='eval_cascade_cm.png'
)

In [ ]:
# ── 7.6  Cascade vs Flat — detailed comparison ────────────────────────────────
f1_flat_best = f1_score(y_test_multi, best_multi_pred,
                        average=None, labels=range(n_classes))
f1_cascade   = f1_score(y_test_multi, y_cascade_multi,
                        average=None, labels=range(n_classes))

x = np.arange(n_classes)
width = 0.38
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, f1_flat_best, width, label=f'Flat: {best_multi_name}', color='#1976D2', alpha=0.9)
ax.bar(x + width/2, f1_cascade,   width, label='Cascade (XGB→LGBM)',      color='#E65100', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(le_target.classes_, rotation=30, ha='right')
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.05)
ax.set_title('Task 2: Flat vs Cascade — Per-Class F1', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'eval_flat_vs_cascade.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'Flat best  Macro F1 : {multi_summary[best_multi_name][0]:.4f}')
print(f'Cascade    Macro F1 : {cascade_macro_f1:.4f}')
delta = multi_summary[best_multi_name][0] - cascade_macro_f1
winner = 'Flat' if delta > 0 else 'Cascade'
print(f'Winner: {winner} by {abs(delta):.4f}')

---
## 8 · Submission Files

In [ ]:
# ── 8.1  Task 1 — Binary submission ──────────────────────────────────────────
print('─' * 55)
print('  TASK 1 — Binary Classification Submission')
print('─' * 55)
print(f'  Model     : XGBoost (threshold-tuned t={best_t_bin:.2f})')
print(f'  F1 Score  : {best_f1_bin:.4f}')
print(f'  ROC-AUC   : {roc_auc_score(y_test_bin, y_prob_bin_xgb):.4f}')

sub1 = save_submission(
    ids=range(len(y_pred_bin_tuned)),
    predictions=y_pred_bin_tuned,
    path=SUB_BINARY,
    task_label='Task 1 (Binary)'
)

In [ ]:
# ── 8.2  Task 2 — Multi-class submission ─────────────────────────────────────
# Use the best-performing approach from Section 7
print('─' * 55)
print('  TASK 2 — Multi-Class Classification Submission')
print('─' * 55)
print(f'  Approach   : {best_multi_name}')
print(f'  Macro F1   : {multi_summary[best_multi_name][0]:.4f}')

# Decode numeric predictions back to string labels
final_multi_preds_labels = le_target.inverse_transform(best_multi_pred)

sub2 = save_submission(
    ids=range(len(final_multi_preds_labels)),
    predictions=final_multi_preds_labels,
    path=SUB_MULTI,
    task_label='Task 2 (Multi-Class)'
)

In [ ]:
# ── 8.3  Predicted vs actual distribution ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
pred_dist   = sub2['prediction'].value_counts().sort_values(ascending=True)
actual_dist = pd.Series(y_test_cat).value_counts().reindex(pred_dist.index).fillna(0)

x = np.arange(len(pred_dist))
w = 0.38
ax.barh(x - w/2, actual_dist.values, w, label='Actual',    color='#42A5F5', alpha=0.85)
ax.barh(x + w/2, pred_dist.values,   w, label='Predicted', color='#EF5350', alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(pred_dist.index)
ax.set_xlabel('Count')
ax.set_title('Task 2: Actual vs Predicted Distribution', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'submission_multi_dist.png', dpi=130, bbox_inches='tight')
plt.show()

---
## 9 · Findings

In [ ]:
# ── 9.1  Final summary ────────────────────────────────────────────────────────
print('=' * 65)
print('FINDINGS SUMMARY')
print('=' * 65)

# Q1: Does cascade beat flat?
best_flat_f1 = multi_summary[best_multi_name][0]
cascade_delta = cascade_macro_f1 - best_flat_f1
print('\nQ1 — Hierarchical cascade vs best flat approach (Task 2 Macro F1):')
print(f'   Cascade               : {cascade_macro_f1:.4f}')
print(f'   Best flat ({best_multi_name[:30]}): {best_flat_f1:.4f}')
if cascade_delta > 0:
    print(f'   ✅ Cascade WINS by {cascade_delta:+.4f}')
    print('   Error propagation from Stage 1 is outweighed by specialised Stage 2 training.')
else:
    print(f'   ❌ Cascade LOSES by {cascade_delta:.4f}')
    print('   Error propagation from Stage 1 (missed attacks) hurts macro F1.')
    print('   Flat models with per-class threshold tuning are more robust.')

# Q2: Hardest classes
print('\nQ2 — Best single model and hardest attack categories:')
print(f'   Best model: {best_multi_name}')
print(f'   Macro F1  : {best_flat_f1:.4f}')
per_class_f1 = f1_score(y_test_multi, best_multi_pred,
                        average=None, labels=range(n_classes))
sorted_classes = sorted(zip(le_target.classes_, per_class_f1), key=lambda x: x[1])
print('   Per-class F1 (worst first):')
for cls, f in sorted_classes:
    bar = '█' * int(f * 20)
    note = '  ← still hardest' if f < 0.30 else ''
    print(f'   {cls:15s}: {f:.3f}  {bar}{note}')

# Q3: Recommended submission strategy
print('\nQ3 — Recommended final submission strategy:')
print(f'   Task 1: XGBoost binary (threshold={best_t_bin:.2f}) → F1={best_f1_bin:.4f}')
print(f'   Task 2: {best_multi_name} → Macro F1={best_flat_f1:.4f}')
print()
print('   Further improvements to try (if time permits):')
print('   • Optuna hyperparameter search (20 trials) on best model')
print('   • Focal loss in XGBoost custom objective for rare classes')
print('   • Increase ADASYN samples for Analysis/Backdoor/Worms specifically')
print('   • Cascade with lower Stage-1 threshold (try 0.20) to recover more attacks')

In [ ]:
# ── 9.2  Final comparison table ───────────────────────────────────────────────
print('\nFINAL RESULTS TABLE')
print('=' * 80)

final_table = [
    ['XGBoost Binary (Task 1, v1 baseline)',     '0.9303',  '—',      '—'],
    [f'XGBoost Binary (Task 1, t={best_t_bin:.2f})', f'{best_f1_bin:.4f}', '—', '—'],
    ['XGBoost flat multi-class (v1 baseline)',   '—', '0.4818',  'SMOTE'],
    ['XGBoost flat (v2, ADASYN)',                '—', f'{xgb_multi_macro_f1:.4f}', 'ADASYN + class weights'],
    ['LightGBM flat',                            '—', f'{lgbm_macro_f1:.4f}', 'class_weight=balanced'],
    ['CatBoost flat',                            '—', f'{cat_macro_f1:.4f}', 'class_weights'],
    ['Stacking (LGBM+XGB+Cat→LR)',               '—', f'{stack_macro_f1:.4f}', '3-fold CV'],
    [f'{best_flat_name} + per-class thresh',     '—', f'{tuned_macro_f1:.4f}', 'per-class t tuning'],
    ['Cascade (XGB Stage1 → LGBM Stage2)',       '—', f'{cascade_macro_f1:.4f}', f't={CASCADE_THRESHOLD}'],
]

ft_df = pd.DataFrame(final_table,
    columns=['Model / Approach', 'Task1 F1', 'Task2 Macro F1', 'Notes'])
print(ft_df.to_string(index=False))

print(f'\n✅ Submission files:')
print(f'   {SUB_BINARY}')
print(f'   {SUB_MULTI}')